# Phase 5b - Artifact-free injection, high-count regime
Phase 5: the additive Gaussian bump forced the count but tinted the image (color DC bias) and lost on the easy 1-5 set (baseline ceiling). Here we test **artifact-free** schemes - `noise_boost` (amplify in-box variance) and `gaussian_noise` (add Gaussian-enveloped fresh noise, zero-mean) - in the **high-count regime (4-8)** where baseline collapses.

**Runtime:** GPU.

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os, yaml, torch
import matplotlib.pyplot as plt
from src.prompts import build_prompt
from src.pipeline import load_sdxl
from src.noise_layout import count_aware_latent
from src.detector import Detector
from src.scoring import count_from_detections
from src.config import load_config

In [ ]:
cfg = load_config('configs/phase5b.yaml')
raw = yaml.safe_load(open('configs/phase5b.yaml'))
schemes = raw['schemes']; obj = cfg.objects[0]
pk = dict(gamma=raw['gamma'], omega=raw['omega'], alpha=raw['alpha'],
          beta=raw['beta'], fill=raw['box_fill'])
pipe = load_sdxl(); det = Detector()
SS = pipe.unet.config.sample_size
def cnt(img):
    return count_from_detections(det.detect(img, [obj]), obj, cfg.score_threshold)
def base_latent(seed):
    g = torch.Generator(device='cpu').manual_seed(seed)
    z = torch.randn((1, pipe.unet.config.in_channels, SS, SS), generator=g)
    return z.to(pipe.device, pipe.dtype)
def gen_latent(prompt, lat):
    return pipe(prompt, latents=lat, num_inference_steps=cfg.num_inference_steps).images[0]

In [ ]:
rows = []
for N in cfg.counts:
    prompt = build_prompt(N, obj)
    for seed in cfg.seeds:
        base = base_latent(seed)
        rows.append({'N': N, 'seed': seed, 'scheme': 'baseline',
                     'rendered': cnt(gen_latent(prompt, base.clone()))})
        for sch in schemes:
            lat = count_aware_latent(base, N, scheme=sch, noise_seed=seed, **pk)
            rows.append({'N': N, 'seed': seed, 'scheme': sch,
                         'rendered': cnt(gen_latent(prompt, lat))})
    print(f'N={N} done')
df = pd.DataFrame(rows)
df['correct'] = df['rendered'] == df['N']
os.makedirs('results', exist_ok=True)
df.to_csv('results/phase5b_counts.csv', index=False)
df.groupby('scheme').agg(exact_acc=('correct', 'mean'),
                         mae=('rendered', lambda s: (df.loc[s.index, 'N'] - s).abs().mean()))

In [ ]:
order = ['baseline'] + schemes
acc = df.groupby('scheme')['correct'].mean().reindex(order)
mae = df.assign(err=(df['N'] - df['rendered']).abs()).groupby('scheme')['err'].mean().reindex(order)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(acc.index, acc.values, color=['gray', 'C0', 'C1', 'C2'])
axes[0].set_ylabel('exact-count accuracy'); axes[0].set_ylim(0, 1)
axes[0].set_title('Exact accuracy (high-count regime)')
axes[1].bar(mae.index, mae.values, color=['gray', 'C0', 'C1', 'C2'])
axes[1].set_ylabel('MAE of count'); axes[1].set_title('Count MAE (lower better)')
plt.tight_layout()
plt.savefig('results/phase5b_accuracy.png', dpi=100, bbox_inches='tight'); plt.show()
print('exact acc:\n', acc, '\nMAE:\n', mae)

In [ ]:
# Eyeball: baseline vs each artifact-free scheme at a high count.
N0, seed0 = 6 if 6 in cfg.counts else cfg.counts[-1], cfg.seeds[0]
prompt0 = build_prompt(N0, obj); base = base_latent(seed0)
cells = [('baseline', base.clone())] + \
        [(s, count_aware_latent(base, N0, scheme=s, noise_seed=seed0, **pk)) for s in schemes]
fig, axes = plt.subplots(1, len(cells), figsize=(3.2 * len(cells), 3.4))
for ax, (name, lat) in zip(axes, cells):
    im = gen_latent(prompt0, lat)
    ax.imshow(im); ax.axis('off')
    ax.set_title(f'{name} | asked {N0} -> {cnt(im)}', fontsize=9)
plt.tight_layout()
plt.savefig('results/phase5b_eyeball.png', dpi=90, bbox_inches='tight'); plt.show()

## How to read this
- **An artifact-free scheme (noise_boost / gaussian_noise) beats baseline on exact accuracy or MAE at high counts, in COHERENT images** = a working training-free mitigation where it matters -> the arc closes with a real fix.
- **Eyeball is decisive:** do noise_boost / gaussian_noise images show the requested number of *photorealistic* cats (no green icons / box artifacts like plain gaussian)? Coherent + correct count = success.
- **Still no gain / still degraded** = training-free noise injection is insufficient on SDXL even at high counts; a clean fix needs the fine-tuning the paper pairs with it. Honest negative; the causal finding stands. Then consolidate for the writeup.